In [2]:
import pygame as pg
import numpy as np
import matplotlib.pyplot as plt

pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [13]:
class vertice:
    def __init__(self, pos, fixed):
        self.curr_pos = np.array(pos, dtype=float)
        self.last_pos = np.array(pos, dtype=float)
        self.fixed = fixed

class edge:
    def __init__(self, p1, p2, rigidness:float = 1):
        self.p1 = p1
        self.p2 = p2
        self.rigid = rigidness
        self.rest_length = np.linalg.norm(p1.curr_pos - p2.curr_pos)

def update_point(p, g:np.ndarray = np.array([0, 500]), dt = 1/60, damp = 0.99):
    if p.fixed:
        return False

    v = p.curr_pos - p.last_pos

    p.last_pos = p.curr_pos.copy()

    p.curr_pos += v * damp
    p.curr_pos += g * dt * dt

    return True

def solve_edge(edge, corr_coef:float = 0.5):
    p1 = edge.p1
    p2 = edge.p2

    delta = p2.curr_pos - p1.curr_pos

    distance = np.linalg.norm(delta)

    if distance==0:
        return False

    diff = distance - edge.rest_length

    direction = delta/distance

    push = direction * diff * corr_coef

    if not p1.fixed:
        p1.curr_pos += push

    if not p2.fixed:
        p2.curr_pos -= push

    return True

def build_mesh(n:int, m:int, spacing:int = 20, style:str = 'clothesline'):

    pos_matrix = []

    for i in range(n+1):
        
        curr_line = []
        
        for j in range(m+1):

           curr_line.append((i * spacing, j * spacing))

        pos_matrix.append(curr_line)


    for row_id, i in enumerate(pos_matrix):

        if style == 'flag':

            i[0] = vertice(i[0], fixed=True)

            for idx in range(1, len(i)):

                i[idx] = vertice(i[idx], fixed=False)

        elif style == 'sidehang':

            i[0] = vertice(i[0], fixed=True)
            i[m] = vertice(i[m], fixed=True)

            for idx in range(1, len(i)-1):

                i[idx] = vertice(i[idx], fixed=False)
        
        elif style == 'clothesline':

            if row_id == 0:

                for idx in range(len(i)):

                    i[idx] = vertice(i[idx], fixed=True)

                continue

            for idx in range(len(i)):

                i[idx] = vertice(i[idx], fixed=False)

        else:
            raise NotImplementedError('Não fiz essa opção ainda meu fi')

    return pos_matrix

edges = []
n, m = (10, 10)
vertices = build_mesh(n, m)

for i in range(n+1):
    for j in range(m+1):

        if i < n:
            edges.append(
                edge(vertices[i][j], vertices[i+1][j])
            )

        if j < m:
            edges.append(
                edge(vertices[i][j], vertices[i][j+1])
            )

pg.init()
running = True
screen = pg.display.set_mode((1280, 720))
clock = pg.time.Clock()
edge_iter = 10

while running:
    # poll for events
    # pg.QUIT event means the user clicked X to close your window
    for event in pg.event.get():
        if event.type == pg.QUIT:
            running = False

    screen.fill('black')

    for i in vertices:
        for k in i:
            update_point(k)

    for _ in range(edge_iter):

        for i in edges:
            solve_edge(i)

    for e in edges:

        pg.draw.line(
            screen,
            (255,255,255),
            e.p1.curr_pos,
            e.p2.curr_pos
        )

    pg.display.flip()
    clock.tick(60)



KeyboardInterrupt: 